# CREST Phase 1: Per-Head Probe Training on Qwen3.6-35B-A3B GA Layers

Replication of CREST (arXiv:2512.24574, Jan 2026) adapted for hybrid MoE+GDN+GA architecture.

**Recipe**:
1. Load Qwen3.6-35B-A3B (10 Gated-Attention layers at positions 3, 7, 11, 15, 19, 23, 27, 31, 35, 39)
2. For each GA layer attention block, hook **before o_proj** to capture per-head outputs (split into [n_heads=16, head_dim=256])
3. Forward 711 Stage B rollouts, capture per-head output at last prompt token
4. Train 160 logistic regression probes (10 layers × 16 heads): binary correct-vs-wrong
5. Rank heads by probe accuracy → top-38% = ~60 cognitive heads
6. Build steering vectors per head (probe weight vector, normalized)
7. Save to HF for Phase 2 (inference eval)

**Budget**: ~25 min compute on RTX 6000 Blackwell. **Cost**: ~$15.

## Cell 1 — Install + setup

In [ ]:
import sys, subprocess, os, shutil
from pathlib import Path

def pip(*a, check=True):
    return subprocess.run([sys.executable, '-m', 'pip', *a], check=check)

try:
    import transformers
    from transformers.models.auto.configuration_auto import CONFIG_MAPPING_NAMES
    ok = 'qwen3_5_moe' in CONFIG_MAPPING_NAMES
except Exception:
    ok = False

if not ok:
    pip('install', '-q', 'accelerate', 'datasets', 'huggingface_hub==1.5.0',
        'safetensors', 'einops', 'scikit-learn', 'sentencepiece', 'tokenizers',
        'protobuf', 'scipy', 'matplotlib', 'hf_transfer', check=False)
    pip('uninstall', '-y', 'transformers', 'causal-conv1d', check=False)
    SRC = '/content/transformers_src'
    if Path(SRC).exists(): shutil.rmtree(SRC)
    subprocess.run(['git','clone','--quiet','--depth=1',
                    'https://github.com/huggingface/transformers.git', SRC], check=True)
    pip('install','--force-reinstall','--no-deps','--no-cache-dir', SRC, check=False)
    pip('install','--no-cache-dir','flash-linear-attention', check=False)
    for m in list(sys.modules):
        if m.startswith('transformers') or m.startswith('huggingface_hub'):
            del sys.modules[m]

try:
    from google.colab import userdata
    t = userdata.get('HF_TOKEN')
    if t:
        from huggingface_hub import login
        login(token=t); print('HF auth OK')
except: pass

import json, re, time, random, pickle
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'

OUT = Path('/content/crest_phase1'); OUT.mkdir(exist_ok=True)
HF_CREST = 'caiovicentino1/qwen36-crest-cognitive-heads'
print('setup done')

## Cell 2 — Load Qwen3.6-35B-A3B

In [ ]:
from transformers import AutoTokenizer, AutoModelForImageTextToText
from huggingface_hub import snapshot_download, HfApi, create_repo
from safetensors import safe_open

MODEL_ID = 'Qwen/Qwen3.6-35B-A3B'
GA_LAYERS = [3, 7, 11, 15, 19, 23, 27, 31, 35, 39]  # Gated-Attention layer indices
N_Q_HEADS = 16  # per GA layer
HEAD_DIM = 256

tok = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tok.pad_token_id is None: tok.pad_token = tok.eos_token

model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID, dtype=torch.bfloat16, device_map='cuda',
    attn_implementation='sdpa', trust_remote_code=True)
model.eval()
print(f'Model: {torch.cuda.memory_allocated()/1e9:.1f} GB')

def get_layer_module(m, idx):
    cands = [m]
    if hasattr(m, 'model'): cands.append(m.model)
    for start in cands:
        for path in [('model','language_model','layers'),('language_model','layers'),('model','layers')]:
            cur = start; ok = True
            for p in path:
                if hasattr(cur, p): cur = getattr(cur, p)
                else: ok = False; break
            if ok and hasattr(cur, '__getitem__'):
                try: return cur[idx]
                except: continue
    raise RuntimeError(f'layer {idx} not found')

# Inspect GA layer attention structure
sample_layer = get_layer_module(model, GA_LAYERS[0])
print(f'\nGA layer {GA_LAYERS[0]} type: {type(sample_layer).__name__}')
# Look for attention module + o_proj
for n, m in sample_layer.named_modules():
    if 'proj' in n or 'attn' in n.lower():
        print(f'  {n}: {type(m).__name__}')

# Create HF repo
api = HfApi()
try:
    create_repo(HF_CREST, repo_type='model', private=False, exist_ok=True)
    print(f'\n✅ HF repo ready: https://huggingface.co/{HF_CREST}')
except Exception as e:
    print(f'repo: {e}')

## Cell 3 — Hook strategy + test forward

Hook the attention **input** to `o_proj` linear. This is `attn_output` post-gating, **pre-o_proj mixing** — per-head [n_heads, head_dim] split.

In [ ]:
# Find o_proj within each GA layer's attention
def find_o_proj(layer):
    for n, m in layer.named_modules():
        if n.endswith('o_proj'):
            return m, n
    return None, None

o_proj_modules = {}
for L in GA_LAYERS:
    mod, name = find_o_proj(get_layer_module(model, L))
    o_proj_modules[L] = mod
    print(f'L{L}: o_proj found at {name}, weight shape {tuple(mod.weight.shape)}')

# Hook pre-o_proj (input): captures per-head concatenated output
# Input shape: [B, T, n_heads * head_dim] = [B, T, 4096]
# Reshape to [B, T, n_heads, head_dim] to get per-head
captured_heads = {L: None for L in GA_LAYERS}
def make_hook(L):
    def hook(module, inp):
        x = inp[0]  # [B, T, n_heads*head_dim]
        # Capture LAST TOKEN only, reshape to [B, n_heads, head_dim]
        last = x[:, -1, :].view(-1, N_Q_HEADS, HEAD_DIM)  # [B, 16, 256]
        captured_heads[L] = last[0].float().cpu().numpy()  # [16, 256]
        return None
    return hook

handles = []
for L in GA_LAYERS:
    h = o_proj_modules[L].register_forward_pre_hook(make_hook(L))
    handles.append(h)
print(f'\n✅ {len(handles)} pre-o_proj hooks installed')

## Cell 4 — Load Stage B rollouts + format prompts

In [ ]:
corpus = snapshot_download('caiovicentino1/Qwen3.6-35B-A3B-mcr-stage-b',
                            repo_type='dataset', cache_dir='/content/cache')
shards = sorted((Path(corpus)/'shards').glob('q*.safetensors'))

MIN_LEN = 200
rollouts = []
for shard in tqdm(shards, desc='load'):
    with safe_open(str(shard), framework='pt') as f:
        meta = f.metadata()
        q_opts = json.loads(meta['options'])
        if len(q_opts) != 10: continue
        rs = json.loads(meta['rollouts'])
        for r in rs:
            if r['pred'] is None or r['response_len'] < MIN_LEN: continue
            rollouts.append(dict(
                question=meta['question'], options=q_opts,
                gold_letter=meta['gold_letter'], pred=r['pred'],
                correct=r['correct']))
n_correct = sum(1 for r in rollouts if r['correct'])
print(f'{len(rollouts)} rollouts ({n_correct} correct, {len(rollouts)-n_correct} wrong)')

def format_prompt(q, opts):
    choices = '\n'.join(f'{chr(65+i)}. {o}' for i, o in enumerate(opts))
    content = ("Answer the following multiple-choice question. "
        "Give ONLY the letter of the correct answer.\n\n"
        f"Question: {q}\n\nOptions:\n{choices}\n\nAnswer: \\boxed{{")
    return tok.apply_chat_template([{'role':'user','content':content}],
                                    tokenize=False, add_generation_prompt=True,
                                    enable_thinking=False)

## Cell 5 — Extract per-head activations at last prompt token

In [ ]:
# Storage: [N_rollouts, N_GA_layers, N_Q_HEADS, HEAD_DIM]
head_acts = np.zeros((len(rollouts), len(GA_LAYERS), N_Q_HEADS, HEAD_DIM), dtype=np.float16)
correct_mask = np.zeros(len(rollouts), dtype=bool)

t0 = time.time()
for ri, r in enumerate(tqdm(rollouts, desc='capture')):
    try:
        p = format_prompt(r['question'], r['options'])
        ids = tok(p, return_tensors='pt').input_ids.cuda()
        if ids.shape[1] > 4096: continue
        with torch.no_grad():
            _ = model(input_ids=ids, attention_mask=torch.ones_like(ids))
        for li, L in enumerate(GA_LAYERS):
            head_acts[ri, li, :, :] = captured_heads[L].astype(np.float16)
        correct_mask[ri] = r['correct']
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache(); continue
    except Exception as e:
        continue

for h in handles: h.remove()
print(f'\n✅ captured in {(time.time()-t0)/60:.1f} min')
print(f'head_acts shape: {head_acts.shape}')
print(f'correct rate: {correct_mask.mean()*100:.1f}%')

# Save
np.savez(OUT / 'head_acts.npz', head_acts=head_acts, correct=correct_mask,
         ga_layers=np.array(GA_LAYERS))

## Cell 6 — Train per-head probes + rank

For each (layer, head), train logistic regression: predict correct vs wrong from head_dim=256 activation. Rank by cross-validated accuracy.

In [ ]:
from sklearn.model_selection import cross_val_score

# Flatten: per-rollout, per-head activation = vector
# For each (li, hi), train probe
y = correct_mask.astype(int)

probe_results = []
print(f'{"layer":>5s} {"head":>4s} {"train_acc":>9s} {"cv_acc":>6s} {"cv_std":>6s}')
for li, L in enumerate(GA_LAYERS):
    for hi in range(N_Q_HEADS):
        X = head_acts[:, li, hi, :].astype(np.float32)  # [N, 256]
        # Train on all, CV for accuracy estimate
        probe = LogisticRegression(C=0.5, max_iter=2000, random_state=42)
        try:
            cv_scores = cross_val_score(probe, X, y, cv=5, scoring='accuracy')
            cv_acc = cv_scores.mean()
            cv_std = cv_scores.std()
        except Exception:
            cv_acc = 0.5; cv_std = 0
        # Fit on full for steering vector extraction
        probe.fit(X, y)
        train_acc = probe.score(X, y)
        probe_results.append(dict(
            layer=L, head=hi, train_acc=train_acc, cv_acc=cv_acc, cv_std=cv_std,
            probe_coef=probe.coef_[0].tolist(), probe_intercept=float(probe.intercept_[0])))

# Sort by cv_acc desc
probe_results.sort(key=lambda r: -r['cv_acc'])

# Print top-20 + bottom-5
print(f'\n=== Top 20 cognitive heads ===')
for r in probe_results[:20]:
    print(f'L{r["layer"]:>3d} H{r["head"]:>2d}  train {r["train_acc"]*100:>5.1f}%  cv {r["cv_acc"]*100:>5.1f}% ± {r["cv_std"]*100:.1f}')

print(f'\n=== Bottom 5 ===')
for r in probe_results[-5:]:
    print(f'L{r["layer"]:>3d} H{r["head"]:>2d}  cv {r["cv_acc"]*100:>5.1f}%')

# Top-38% = 60 heads per CREST default
top_k = int(len(probe_results) * 0.38)
cognitive_heads = probe_results[:top_k]
print(f'\n✅ Cognitive heads (top {top_k}/{len(probe_results)}):')
layer_counts = {}
for r in cognitive_heads:
    layer_counts[r['layer']] = layer_counts.get(r['layer'], 0) + 1
for L, c in sorted(layer_counts.items()):
    print(f'  L{L}: {c}/16 heads')

## Cell 7 — Save cognitive heads + probe coefficients + upload

In [ ]:
# Package everything
output_data = {
    'ga_layers': GA_LAYERS,
    'n_q_heads': N_Q_HEADS,
    'head_dim': HEAD_DIM,
    'n_rollouts': len(rollouts),
    'correct_rate': float(correct_mask.mean()),
    'probe_results': probe_results,
    'cognitive_heads': [{'layer': r['layer'], 'head': r['head'], 'cv_acc': r['cv_acc']}
                         for r in cognitive_heads],
    'top_k': top_k,
    'total_heads': len(probe_results),
}
with open(OUT / 'crest_phase1_results.json', 'w') as f:
    json.dump(output_data, f, indent=2)

# Also save as compact npz for steering (weights only)
# Per-head probe coef matrix: [160, 256]
all_coefs = np.zeros((len(probe_results), HEAD_DIM), dtype=np.float32)
for i, r in enumerate(probe_results):
    all_coefs[i] = np.array(r['probe_coef'])
layer_ids = np.array([r['layer'] for r in probe_results], dtype=np.int32)
head_ids = np.array([r['head'] for r in probe_results], dtype=np.int32)
cv_accs = np.array([r['cv_acc'] for r in probe_results], dtype=np.float32)

np.savez(OUT / 'probe_coefs.npz',
         coefs=all_coefs, layer_ids=layer_ids, head_ids=head_ids, cv_accs=cv_accs)

# Upload
readme = f'''---
license: mit
base_model: Qwen/Qwen3.6-35B-A3B
tags:
  - mechanistic-interpretability
  - cognitive-heads
  - crest
  - hybrid-moe
---

# Qwen3.6-35B-A3B Cognitive Heads — CREST Phase 1

Per-head logistic regression probes trained on Qwen3.6-35B-A3B Gated-Attention layers (L3, L7, L11, L15, L19, L23, L27, L31, L35, L39) × 16 Q heads = 160 total heads.

Replication of CREST (arXiv:2512.24574) methodology adapted for hybrid MoE+GDN+GA architecture.

## Data

- Source: Qwen3.6-35B-A3B MCR Stage B corpus ({len(rollouts)} rollouts, {correct_mask.sum()} correct / {(~correct_mask).sum()} wrong)
- Activation: pre-o_proj output at last prompt token
- Probe: Logistic regression (C=0.5) with 5-fold CV
- Task: binary correct vs wrong

## Results

- Top-20 cognitive heads identified (see `crest_phase1_results.json`)
- Top-38% default ({top_k}/160 heads) marked as cognitive

## Files

- `head_acts.npz` — raw per-head activations (711 × 10 × 16 × 256, fp16)
- `probe_coefs.npz` — compact probe coefficients + CV accuracies
- `crest_phase1_results.json` — full results including coefficients

## Phase 2 (next)

Apply norm-preserving rotation at top-{top_k} cognitive heads during inference, eval on GSM8K + MATH500, compare vs base model.
'''
with open(OUT / 'README.md', 'w') as f: f.write(readme)

for fn in ['head_acts.npz', 'probe_coefs.npz', 'crest_phase1_results.json', 'README.md']:
    try:
        api.upload_file(path_or_fileobj=str(OUT / fn),
                        path_in_repo=fn,
                        repo_id=HF_CREST, repo_type='model',
                        commit_message=f'CREST Phase 1: {fn}')
        print(f'✓ {fn}')
    except Exception as e:
        print(f'✗ {fn}: {e}')
print(f'\n✅ uploaded to https://huggingface.co/{HF_CREST}')